# 门控残差融合适配器：JupyterLab 环境 + MATH 训练

在远程 JupyterLab 服务器上搭建环境，并用自己的 `train_fusion.py` 在 MATH 数据集上训练 22M 旁路适配器。

流程：建环境、装依赖、检查 GPU、加载并转换 MATH、跑训练、查看 loss 图与产物。

注意：只用 MATH 的 train 划分(7.5k)训练，test 划分(5k)留作后续泛化评测，避免数据泄漏。

## 1. 创建并激活 Conda 环境

In [ ]:
# 创建 Python 3.11 环境(与本地开发环境一致)
!conda create -n bes python=3.11 -y

# 安装 ipykernel 并注册到 JupyterLab，以便使用该内核
!conda run -n bes pip install ipykernel
!conda run -n bes python -m ipykernel install --user --name bes --display-name "Python (bes)" 

注册完成后，在 JupyterLab 右上角把内核切换为 **Python (bes)**，后续所有 cell 都跑在这个环境里。

## 2. 安装 PyTorch、Transformers、Datasets 等依赖

In [ ]:
# PyTorch：按服务器 CUDA 版本选择 index，下面给 CUDA 12.1 示例
# (不确定就执行 nvidia-smi 看版本，或直接 pip install torch 自动带 CUDA)
!conda run -n bes pip install torch --index-url https://download.pytorch.org/whl/cu121

# 其余依赖：transformers 5.x / accelerate / modelscope 定位模型 / matplotlib 画 loss 图 / datasets 下载 MATH
!conda run -n bes pip install transformers accelerate modelscope matplotlib datasets

## 3. 检查 GPU 并进入项目目录

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("torch:", torch.__version__)

# 进入已上传的项目目录(本项目需要 train_fusion.py / main.py 等)
%cd /root/BES

若 `CUDA available` 为 False，说明 torch 没装 CUDA 版本，回到第 2 步按实际 CUDA 版本换 index。

## 4. 加载 MATH 数据集

In [ ]:
from datasets import load_dataset

# Hendrycks MATH 的 train 划分(7500 题)
ds = load_dataset("hendrycks/competition_math", split="train", trust_remote_code=True)
print("样本数:", len(ds))
print("字段:", list(ds[0].keys()))
print("problem 示例:", ds[0]["problem"][:150])
print("solution 示例:", ds[0]["solution"][:150])

## 5. 预处理：转成 train_fusion.py 需要的 jsonl 格式

In [ ]:
import json

# train_fusion.py 的 load_texts 每行读 {"text": ...}，tokenize 与截断由脚本内部完成。
with open("data/math_train.jsonl", "w", encoding="utf-8") as f:
    for ex in ds:
        text = ex["problem"].strip() + "\n解题思路:" + ex["solution"].strip()
        f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
print("已生成 data/math_train.jsonl")

## 6. 加载预训练模型与训练配置（由 train_fusion.py 完成）

In [ ]:
# train_fusion.py 内部会加载 Qwen3-4B(主脑) 与 Qwen3-0.6B(旁路宿主) 并全部冻结，
# 只训练 22M 旁路参数。训练超参走命令行，先看有哪些选项：
!conda run -n bes python core_training/train_fusion.py --help

## 7. 执行训练

In [ ]:
# 小规模试跑(约 200 条，几分钟)：先跑这个，确认能跑通且评估行"增益"为正
!conda run -n bes python core_training/train_fusion.py --data data/math_train.jsonl --max_samples 200 --epochs 1 --eval_every 20

# 确认无误后，取消下面注释跑正式训练(全量 7.5k，3 epoch，早停耐心 3)
# !conda run -n bes python core_training/train_fusion.py --data data/math_train.jsonl --epochs 3 --eval_every 100 --patience 3

## 8. 保存与验证

In [ ]:
from IPython.display import Image, display

# 产物：loss 图 / 最终权重 / 最优权重
#   cache/train_fusion_loss.png  — loss 图
#   cache/fusion_adapter.pt      — 最终权重
#   cache/fusion_adapter.pt.best — 最优权重
display(Image("cache/train_fusion_loss.png"))

# 观察要点：
#   评估行"增益"应大于 0(表示旁路有用)且随训练上升
#   gate 应稳定在 0.5 附近，不跌向 0
# 真正的泛化评测用 MATH 的 test 划分(本 notebook 未使用)，避免数据泄漏